# 01 - Data audit and leakage screen**Run before any model is trained.** Every feature flagged here gets an explicit keep/drop decision recorded in `_LEAKAGE_NOTES.md`.Particular attention to CIC-Bell-DNS2021: its third-party reputation features may partly encode the label. A model scoring 0.99 is usually reading the answer.

In [ ]:
# --- standard header: every notebook starts with exactly this ---from google.colab import drive; drive.mount('/content/drive')REPO = '/content/secure-dns-trust-ai'!git -C {REPO} pull -q 2>/dev/null || git clone -q https://github.com/sandesh20lamichhane/secure-dns-trust-ai.git {REPO}import sys, os; sys.path.insert(0, REPO)os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'%load_ext autoreload%autoreload 2from src.utils import config, manifest, seeds, ioP = config.paths(); config.ensure_tree(P); seeds.set_all(42)print('repo', manifest.git_sha(REPO))

In [ ]:
import pandas as pdfrom src.evaluate import leakagedf = pd.read_parquet(f"{P['data']['interim']}/domains_labelled.parquet")print(df.shape, df['label'].value_counts(normalize=True).round(4).to_dict())

In [ ]:
rep = leakage.report(df)display(rep['single_feature_auc'].head(25))print('CRITICAL (auc >= 0.97):', rep['critical_features'])print(rep['duplicates'])

In [ ]:
display(rep['missingness'].head(20))# Class-dependent missingness leaks the label through NaN patterns alone.

In [ ]:
# Append a decision for each flagged feature - this text becomes methodology prose.entry = '''## Feature: third_party_reputation_score**Why suspicious:** single-feature AUC 0.99 on the training split.**Investigation:** distribution by class; traced to an external blocklist aggregation.**Decision:** EXCLUDED from all primary experiments (quarantined in features.yaml).**Reason:** encodes ground truth indirectly; retaining it would make the reportedperformance a measurement of the blocklist rather than of the model.'''open(P['leakage_notes'], 'a').write(entry)